#GenDEBRIS: U-Net + LoRA for Marine Debris Segmentation
###Implementation with full reproducibility.
###Key results: Encoder-only LoRA (r=16) beats full fine-tuning



In [15]:
# ============================================================================
# SHARED SETUP - Run ONCE at the beginning
# ============================================================================

print("\n" + "="*80)
print("SETUP: DEPENDENCIES & DATA")
print("="*80)

# Install dependencies
!pip install -q transformers peft timm accelerate torch torchvision segmentation-models-pytorch scikit-learn pandas matplotlib seaborn

# Mount drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/harshini2806/lora_paper.git
%cd lora_paper

# Ensure experiment reproducibility
import os
os.makedirs('/content/drive/MyDrive/lora_experiments', exist_ok=True)

# Core imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Any
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
import numpy as np
import time
from collections import Counter
import pandas as pd

# U-Net models
from segmentation_models_pytorch import Unet

# Constants
SEED = 42
BATCH_SIZE = 4
IMG_SIZE = 224
NUM_CLASSES = 11
DEBRIS_ID = 0  # Marine Debris (class 0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
import torch.backends.cudnn as cudnn
import random

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# ============================================================================
# DATA LOADING & CLASS WEIGHTS
# ============================================================================

# Load GenDEBRIS dataset
from dataloader import GenDEBRIS
train_ds = GenDEBRIS(mode='train', path='/content/lora_paper/data')
val_ds = GenDEBRIS(mode='val', path='/content/lora_paper/data')
print(f"Dataset: Train={len(train_ds)}, Val={len(val_ds)}")

# Compute class weights (inverse frequency)
label_counts = Counter()
for _, mask in train_ds:
    valid = mask[mask != -1].flatten()
    label_counts.update(valid.tolist())

total_pixels = sum(label_counts.values())
freqs = np.array([label_counts.get(i, 0) / total_pixels for i in range(NUM_CLASSES)])
class_weights = torch.tensor(1.0 / (freqs + 1e-6), dtype=torch.float32).to(device)
class_weights /= class_weights.sum()

# ============================================================================
# METRICS
# ============================================================================

def compute_metrics(preds: torch.Tensor, labels: torch.Tensor, num_classes: int = NUM_CLASSES) -> Dict[str, float]:
    if preds.dim() == 4:
        preds = torch.argmax(preds, dim=1).cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    valid = labels != -1
    preds_v = preds[valid]
    labels_v = labels[valid]

    # Pixel accuracy
    accuracy = (preds_v == labels_v).sum() / labels_v.size

    # Weighted F1
    f1 = f1_score(labels_v, preds_v, average='weighted', zero_division=0)

    # mIoU (mean over present classes)
    present_classes = np.unique(labels_v)
    ious = []
    for cls in present_classes:
        pred_m = (preds_v == cls)
        true_m = (labels_v == cls)
        inter = (pred_m & true_m).sum()
        union = (pred_m | true_m).sum()
        ious.append(inter / union if union > 0 else 0.0)
    mean_iou = np.mean(ious)

    # Debris IoU (class 0)
    debris_mask = (labels_v == DEBRIS_ID)
    debris_pred = (preds_v == DEBRIS_ID)
    inter_debris = (debris_mask & debris_pred).sum()
    union_debris = (debris_mask | debris_pred).sum()
    debris_iou = inter_debris / union_debris if union_debris > 0 else 0.0

    return {
        'Accuracy': float(accuracy),
        'F1': float(f1),
        'mIoU': float(mean_iou),
        'Debris_IoU': float(debris_iou),
    }

# ============================================================================
# TRAINING
# ============================================================================

def train(model: nn.Module, name: str, epochs: int = 70, lr: float = None) -> Dict[str, Any]:
    print(f"\n{'#'*70}")
    print(f"# {name}")
    print("#"*70)

    # Hyperparameters
    if lr is None:
        lr = 3e-4 if 'lora' in name.lower() else 5e-5

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)
    scaler = GradScaler()

    # DataLoaders
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, worker_init_fn=seed_worker, generator=g)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, worker_init_fn=seed_worker, generator=g)

    # Track best per metric (silent)
    bests = {
        'miou': {'value': 0.0, 'metrics': {}},
        'f1': {'value': 0.0, 'metrics': {}},
        'accuracy': {'value': 0.0, 'metrics': {}},
        'debris': {'value': 0.0, 'metrics': {}}
    }

    start_time = time.time()

    for epoch in range(epochs):
        # Training
        model.train()
        total_loss = 0.0
        for imgs, masks in train_dl:
            if imgs.dim() == 4 and imgs.shape[-1] == 11:
                imgs = imgs.permute(0, 3, 1, 2)
            imgs = F.interpolate(imgs, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
            imgs, masks = imgs.float().to(device), masks.long().to(device)

            optimizer.zero_grad()
            with autocast("cuda"):
                logits = model(imgs)
                if logits.shape[2:] != masks.shape[1:]:
                    logits = F.interpolate(logits, masks.shape[1:], mode='bilinear')
                loss = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        scheduler.step()

        # Validation
        model.eval()
        val_metrics = {'mIoU': 0.0, 'F1': 0.0, 'Accuracy': 0.0, 'Debris_IoU': 0.0}

        with torch.no_grad():
            for imgs, masks in val_dl:
                if imgs.dim() == 4 and imgs.shape[-1] == 11:
                    imgs = imgs.permute(0, 3, 1, 2)
                imgs = F.interpolate(imgs, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
                imgs, masks = imgs.float().to(device), masks.long().to(device)

                with autocast("cuda"):
                    logits = model(imgs)
                    if logits.shape[2:] != masks.shape[1:]:
                        logits = F.interpolate(logits, masks.shape[1:], mode='bilinear')

                met = compute_metrics(logits, masks)
                for k in val_metrics:
                    val_metrics[k] += met[k]

        # Average
        for k in val_metrics:
            val_metrics[k] /= len(val_dl)

        # Log
        print(f"Ep {epoch+1:02d} | Loss={total_loss/len(train_dl):.4f} | "
              f"mIoU={val_metrics['mIoU']:.4f} | F1={val_metrics['F1']:.4f} | "
              f"Acc={val_metrics['Accuracy']:.4f} | DebrisIoU={val_metrics['Debris_IoU']:.4f}")

        # Update bests (silent)
        metrics_map = {
            'miou': 'mIoU',
            'f1': 'F1',
            'accuracy': 'Accuracy',
            'debris': 'Debris_IoU'
        }

        for best_key, metric_key in metrics_map.items():
            if val_metrics[metric_key] > bests[best_key]['value']:
                bests[best_key]['value'] = val_metrics[metric_key]
                bests[best_key]['metrics'] = val_metrics.copy()
                torch.save(model.state_dict(),
                          f"/content/drive/MyDrive/lora_experiments/{name}_best_{best_key}.pth")

    # Final summary (one best per metric, no epochs)
    print(f"\n{name} COMPLETE")
    print(f"  Best mIoU:     {bests['miou']['value']:.4f}")
    print(f"  Best F1:       {bests['f1']['value']:.4f}")
    print(f"  Best Accuracy: {bests['accuracy']['value']:.4f}")
    print(f"  Best DebrisIoU:{bests['debris']['value']:.4f}")

    bests['name'] = name
    bests['wall_time_min'] = (time.time() - start_time) / 60
    return bests

print("SETUP COMPLETE - Ready to train models!")



SETUP: DEPENDENCIES & DATA
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'lora_paper'...
remote: Enumerating objects: 4432, done.
remote: Total 4432 (delta 0), reused 0 (delta 0), pack-reused 4432 (from 1)
Receiving objects: 100% (4432/4432), 1.08 GiB | 7.60 MiB/s, done.
Resolving deltas: 100% (2817/2817), done.
Updating files: 100% (4478/4478), done.
/content/lora_paper/lora_paper/lora_paper/lora_paper


Load val set to memory: 100%|██████████| 328/328 [00:08<00:00, 40.20it/s]


Dataset: Train=694, Val=328
SETUP COMPLETE - Ready to train models!


###Creation of U-Net baseline with ResNet50 encoder

In [16]:
def create_unet_baseline() -> nn.Module:
    """Create U-Net baseline with ResNet50 encoder."""
    print("Creating UNet Baseline (ResNet50 encoder)")

    model = Unet(
        encoder_name="resnet50",        # ImageNet-pretrained backbone
        encoder_weights="imagenet",     # Standard pretrained weights
        in_channels=11,                 # Multispectral Sentinel-2 input
        classes=11,                     # 11 semantic classes post-aggregation
    )
    return model


###Creation U-Net with encoder-only LoRA adaptation.
   

In [17]:
from peft import LoraConfig, get_peft_model
import torch.nn as nn

def create_unet_lora(lora_r: int = 16) -> nn.Module:
    """
    Create U-Net with encoder-only LoRA adaptation.

    Key design: LoRA targets only ResNet encoder (frozen base weights),
    decoder and segmentation head remain fully trainable.

    Args:
        lora_r: LoRA rank (r=16 gives ~5% encoder trainable params)

    Returns:
        PEFT-wrapped U-Net model.
    """
    model = create_unet_baseline()
    print(f"Adding encoder-only LoRA (r={lora_r})")

    lora_cfg = LoraConfig(
        r=lora_r,                           # Rank (trainable subspace dim)
        lora_alpha=lora_r * 2,              # Scaling factor
        target_modules=[
            "conv1", "conv2", "conv3", "conv4",  # ResNet conv blocks
            "downsample.0"                       # Skip connections
        ],
        lora_dropout=0.1,                   # Dropout in LoRA layers
        bias="none",                        # No bias adaptation
        modules_to_save=["decoder", "segmentation_head"]  # Fully trainable
    )

    # Apply LoRA ONLY to encoder (key design choice)
    model.encoder = get_peft_model(model.encoder, lora_cfg)
    model.encoder.print_trainable_parameters()

    return model


###BASELINE: Full Fine-Tuning (U-Net ResNet50)

In [ ]:
print("Running U-Net baseline (full fine-tuning)")
baseline_unet = create_unet_baseline().to(device)

result_baseline = train(
    baseline_unet,
    "unet_baseline_r50_fullft_70ep",  # Descriptive name
    epochs=70,
    lr=5e-5
)

pprint("BASELINE COMPLETE:")
print(f"  Best mIoU:     {result_baseline['miou']['value']:.4f}")
print(f"  Best F1:       {result_baseline['f1']['value']:.4f}")
print(f"  Best Accuracy: {result_baseline['accuracy']['value']:.4f}")
print(f"  Best DebrisIoU:{result_baseline['debris']['value']:.4f}")


Running U-Net baseline (full fine-tuning)
Creating UNet Baseline (ResNet50 encoder)

######################################################################
# unet_baseline_r50_fullft_70ep
######################################################################


 ### RANK ABLATION: Encoder-only LoRA

###r=4



In [10]:
print("r=4: Encoder-only LoRA (lr=3e-4)")
lora_r4 = create_unet_lora(lora_r=4).to(device)
result_r4 = train(lora_r4, "lora_encoder_r4_lr3e4_70ep", epochs=70, lr=3e-4)
print("r=4 Encoder-only LoRA COMPLETE:")
print(f"  Best mIoU:     {result_r4['miou']['value']:.4f}")
print(f"  Best F1:       {result_r4['f1']['value']:.4f}")
print(f"  Best Accuracy: {result_r4['accuracy']['value']:.4f}")
print(f"  Best DebrisIoU:{result_r4['debris']['value']:.4f}")


r=4: Encoder-only LoRA (lr=3e-4)
Creating UNet Baseline (ResNet50 encoder)
Adding encoder-only LoRA (r=4)
trainable params: 319,340 || all params: 23,852,460 || trainable%: 1.3388

######################################################################
# lora_encoder_r4_lr3e4_70ep
######################################################################
Ep 01 | Loss=2.4197 | mIoU=0.1076 | F1=0.2040 | Acc=0.1907 | DebrisIoU=0.1119
Ep 02 | Loss=1.9625 | mIoU=0.2033 | F1=0.3377 | Acc=0.3115 | DebrisIoU=0.1250
Ep 03 | Loss=1.7709 | mIoU=0.2208 | F1=0.3987 | Acc=0.3747 | DebrisIoU=0.1685
Ep 04 | Loss=1.5900 | mIoU=0.2720 | F1=0.4327 | Acc=0.4057 | DebrisIoU=0.1360
Ep 05 | Loss=1.4746 | mIoU=0.2748 | F1=0.4781 | Acc=0.4382 | DebrisIoU=0.1473
Ep 06 | Loss=1.2982 | mIoU=0.2273 | F1=0.4253 | Acc=0.3840 | DebrisIoU=0.1977
Ep 07 | Loss=1.3169 | mIoU=0.3151 | F1=0.5300 | Acc=0.4840 | DebrisIoU=0.2331
Ep 08 | Loss=1.2132 | mIoU=0.3370 | F1=0.5533 | Acc=0.5136 | DebrisIoU=0.2071
Ep 09 | Loss=1.2249 | mI

 ### r=8

In [ ]:
print("r=8: Encoder-only LoRA (lr=3e-4)")
lora_r8 = create_unet_lora(lora_r=8).to(device)
result_r8 = train(lora_r8, "lora_encoder_r8_lr3e4_70ep", epochs=70, lr=3e-4)
print("r=8 Encoder-only LoRA COMPLETE:")
print(f"  Best mIoU:     {result_r8['miou']['value']:.4f}")
print(f"  Best F1:       {result_r8['f1']['value']:.4f}")
print(f"  Best Accuracy: {result_r8['accuracy']['value']:.4f}")
print(f"  Best DebrisIoU:{result_r8['debris']['value']:.4f}")


🟡 Creating UNet Baseline (ResNet50 encoder)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

🔷 Adding LoRA adapters with r=8
trainable params: 9,653,203 || all params: 42,200,846 || trainable%: 22.8744

######################################################################
# unet_lora_r8_70ep
######################################################################

Ep 01 | Loss=2.5725 | mIoU=0.0727 | F1=0.1636 | Acc=0.1194
Ep 02 | Loss=2.2137 | mIoU=0.1204 | F1=0.2377 | Acc=0.2111
Ep 03 | Loss=2.0046 | mIoU=0.1321 | F1=0.2514 | Acc=0.2245
Ep 04 | Loss=1.8039 | mIoU=0.1616 | F1=0.3126 | Acc=0.2721
Ep 05 | Loss=1.7334 | mIoU=0.1887 | F1=0.3704 | Acc=0.3160
Ep 06 | Loss=1.5758 | mIoU=0.1891 | F1=0.3510 | Acc=0.3001
Ep 07 | Loss=1.5640 | mIoU=0.1985 | F1=0.3504 | Acc=0.3145
Ep 08 | Loss=1.4758 | mIoU=0.2398 | F1=0.4452 | Acc=0.4187
Ep 09 | Loss=1.3810 | mIoU=0.2625 | F1=0.4531 | Acc=0.4191
Ep 10 | Loss=1.3076 | mIoU=0.2517 | F1=0.4577 | Acc=0.4100
Ep 11 | Loss=1.2443 | mIoU=0.2609 | F1=0.4805 | Acc=0.4421
Ep 12 | Loss=1.1268 | mIoU=0.2578 | F1=0.4819 | Acc=0.4378
Ep 13 | Loss=1.1484

r=16

In [ ]:
print("r=16: Encoder-only LoRA (lr=3e-4)")
lora_r16 = create_unet_lora(lora_r=16).to(device)
result_r16 = train(lora_r16, "lora_encoder_r16_lr3e4_70ep", epochs=70, lr=3e-4)
print("r=16 Encoder-only LoRA COMPLETE:")
print(f"  Best mIoU:     {result_r16['miou']['value']:.4f}")
print(f"  Best F1:       {result_r16['f1']['value']:.4f}")
print(f"  Best Accuracy: {result_r16['accuracy']['value']:.4f}")
print(f"  Best DebrisIoU:{result_r16['debris']['value']:.4f}")


🟡 Creating UNet Baseline (ResNet50 encoder)
🔷 Adding LoRA adapters with r=16
trainable params: 10,291,883 || all params: 42,839,526 || trainable%: 24.0243

######################################################################
# unet_lora_r16_70ep
######################################################################

Ep 01 | Loss=2.7755 | mIoU=0.1271 | F1=0.2793 | Acc=0.2177
Ep 02 | Loss=2.1886 | mIoU=0.1607 | F1=0.3302 | Acc=0.2860
Ep 03 | Loss=2.0272 | mIoU=0.1939 | F1=0.3654 | Acc=0.3167
Ep 04 | Loss=1.8525 | mIoU=0.2189 | F1=0.3846 | Acc=0.3523
Ep 05 | Loss=1.7310 | mIoU=0.2410 | F1=0.4452 | Acc=0.3922
Ep 06 | Loss=1.5884 | mIoU=0.2730 | F1=0.4696 | Acc=0.4278
Ep 07 | Loss=1.5051 | mIoU=0.2835 | F1=0.5047 | Acc=0.4586
Ep 08 | Loss=1.4040 | mIoU=0.2727 | F1=0.4703 | Acc=0.4223
Ep 09 | Loss=1.2716 | mIoU=0.2733 | F1=0.4625 | Acc=0.4157
Ep 10 | Loss=1.1974 | mIoU=0.2901 | F1=0.4908 | Acc=0.4411
Ep 11 | Loss=1.1432 | mIoU=0.2732 | F1=0.5054 | Acc=0.4601
Ep 12 | Loss=1.0508 | mIoU=0.31

###COMPREHENSIVE TABLE

In [ ]:
def get_trainable_info(model) -> dict:
    """Compute trainable parameters (total and encoder-specific)."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    trainable_pct = 100 * trainable_params / total_params

    encoder_params = sum(p.numel() for p in model.encoder.parameters())
    encoder_trainable = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
    encoder_pct = 100 * encoder_trainable / encoder_params if encoder_params > 0 else 0

    return {
        'trainable_params': f"{trainable_params:,}",
        'trainable_pct': f"{trainable_pct:.1f}%",
        'encoder_trainable': f"{encoder_trainable:,}",
        'encoder_pct': f"{encoder_pct:.1f}%"
    }


# Collect results
all_results = [result_baseline, result_r4, result_r8, result_r16]
names = ['Baseline (Full FT)', 'LoRA r=4', 'LoRA r=8', 'LoRA r=16']

print("\nPUBLICATION TABLE 1: Performance vs Efficiency")
comparison_df = pd.DataFrame({
    'Method': names,
    'mIoU': [r['mIoU'] for r in all_results],
    'F1': [r['F1'] for r in all_results],
    'Debris_IoU': [r['Debris_IoU'] for r in all_results],
    'Wall Time (min)': [r['time_min'].round(1) for r in all_results]
})
print(comparison_df.round(4).to_markdown(index=False))

print("\nPUBLICATION TABLE 2: Trainable Parameters")
param_df = pd.DataFrame({
    'Method': names,
    'Total Trainable': [get_trainable_info(baseline_unet)['trainable_params']],
    'Total %': [get_trainable_info(baseline_unet)['trainable_pct']],
    'Encoder Trainable': [get_trainable_info(m)['encoder_trainable'] for m in [lora_r4, lora_r8, lora_r16]],
    'Encoder %': [get_trainable_info(m)['encoder_pct'] for m in [lora_r4, lora_r8, lora_r16]]
})
print(param_df.to_markdown(index=False))
